# CISB5123 Text Analytics
## Lab Assignment 2 — Sentiment Analysis
**Name:** OSAMA MOHAMMED ALI ABDASLAM  
**Student ID:** SW01084153

---
### Discussion: Strengths and Weaknesses of Selected Models

In this assignment, four sentiment classification models were evaluated: **TextBlob**, **VADER** (lexicon-based), **Naive Bayes**, and **Support Vector Machine / SVM** (machine learning-based). TextBlob is simple and requires no training data, but it struggles with neutral sentiment and informal language, often misclassifying sarcasm or domain-specific vocabulary. VADER is specifically tuned for social media and short texts, handling punctuation and capitalization cues well; however, like TextBlob, it performs poorly on neutral reviews and relies on a fixed lexicon that may not generalise to all food-review expressions. Naive Bayes is fast and effective on small datasets, but it assumes feature independence which does not always hold for natural language. SVM with a linear kernel delivers strong performance by finding an optimal decision boundary in high-dimensional feature spaces, yet it can be slower to train and more sensitive to hyperparameter choices. Overall, machine-learning-based approaches outperform lexicon-based ones on labelled datasets because they learn domain-specific patterns directly from the data.

---

## Step 1 — Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('Reviews.csv')
print('Full dataset shape:', df.shape)
df.head(3)

Full dataset shape: (568454, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [2]:
# Sample for faster experimentation (remove / increase for full run)
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

# Map Score → Sentiment label 
# Score 1-2 → negative | Score 3 → neutral | Score 4-5 → positive
def map_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(map_sentiment)
print('Sentiment distribution:')
print(df['Sentiment'].value_counts())

Sentiment distribution:
Sentiment
positive    3919
negative     703
neutral      378
Name: count, dtype: int64


In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'<.*?>', ' ', text)                # remove HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)            # remove non-alpha
    text = re.sub(r'\s+', ' ', text).strip()          # collapse spaces
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['CleanText'] = df['Text'].apply(preprocess)
print('Sample cleaned review:')
print(df['CleanText'].iloc[0])

Sample cleaned review:
tried couple brand gluten free sandwich cooky best bunch crunchy true texture real cooky gluten free might think filling make bit sweet mean satisfied sweet tooth sooner chocolate version glutino good true chocolatey taste something gluten free brand


## Step 2 — Feature Extraction

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

texts  = df['CleanText'].tolist()
labels = df['Sentiment'].tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Bag-of-Words
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train_raw)
X_test_bow  = bow_vectorizer.transform(X_test_raw)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_raw)

print('BoW  train shape:', X_train_bow.shape)
print('TF-IDF train shape:', X_train_tfidf.shape)

BoW  train shape: (4000, 5000)
TF-IDF train shape: (4000, 5000)


## Step 3 — Lexicon-Based Approach (TextBlob & VADER)

In [5]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report
from tabulate import tabulate

# Use raw (uncleaned) test texts for lexicon methods — they rely on punctuation/caps
raw_test_texts = X_test_raw

analyzer = SentimentIntensityAnalyzer()

tb_preds    = []
vader_preds = []

table_data = [["Text (truncated)", "Actual", "TB Polarity", "TB Pred",
               "VADER Compound", "VADER Pred"]]

for text in raw_test_texts:
    # TextBlob
    polarity = TextBlob(text).sentiment.polarity
    tb_label = 'positive' if polarity > 0 else ('negative' if polarity < 0 else 'neutral')
    tb_preds.append(tb_label)

    # VADER
    compound = analyzer.polarity_scores(text)['compound']
    vader_label = 'positive' if compound > 0.05 else ('negative' if compound < -0.05 else 'neutral')
    vader_preds.append(vader_label)

    table_data.append([text[:60] + '...', '', round(polarity, 4), tb_label,
                       round(compound, 4), vader_label])

# Show first 10 rows
print(tabulate(table_data[:11], headers='firstrow', tablefmt='plain'))

Text (truncated)                                                 Actual      TB Polarity  TB Pred      VADER Compound  VADER Pred
almond taste good well flavored cinnamon vanilla flavored de...                   0.1162  positive             0.6124  positive
bishon shitzu puppy love last long rawhide like treat mini s...                   0.3625  positive             0.9201  positive
great price excellent product cat take pill twice day take w...                   0.9     positive             0.872   positive
coffee around refreshing etheopian grind taste good although...                   0.2375  positive             0.0516  positive
husband favorite flavor find anywhere esp since wanted roll ...                   0.5     positive             0.4588  positive
quite tasty mild chocolate flavor good crunch right sweetnes...                   0.5657  positive             0.9485  positive
picky eater successful gastric bypass surgery weight loss ca...                   0.2122  positive    

In [6]:
print('Classification Report — TextBlob:')
print(classification_report(y_test, tb_preds,
                             target_names=['negative', 'neutral', 'positive']))

print('Classification Report — VADER:')
print(classification_report(y_test, vader_preds,
                             target_names=['negative', 'neutral', 'positive']))

Classification Report — TextBlob:
              precision    recall  f1-score   support

    negative       0.51      0.37      0.43       141
     neutral       0.23      0.04      0.07        75
    positive       0.83      0.94      0.89       784

    accuracy                           0.79      1000
   macro avg       0.53      0.45      0.46      1000
weighted avg       0.74      0.79      0.76      1000

Classification Report — VADER:
              precision    recall  f1-score   support

    negative       0.62      0.23      0.33       141
     neutral       0.00      0.00      0.00        75
    positive       0.82      0.96      0.88       784

    accuracy                           0.79      1000
   macro avg       0.48      0.40      0.41      1000
weighted avg       0.73      0.79      0.74      1000



## Step 4 — Machine Learning Approach (Naive Bayes & SVM)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Naive Bayes (BoW features)
nb_clf = MultinomialNB()
nb_clf.fit(X_train_bow, y_train)
nb_preds = nb_clf.predict(X_test_bow)

print('Classification Report — Naive Bayes (BoW):')
print(classification_report(y_test, nb_preds,
                             target_names=['negative', 'neutral', 'positive']))

Classification Report — Naive Bayes (BoW):
              precision    recall  f1-score   support

    negative       0.63      0.52      0.57       141
     neutral       0.41      0.15      0.22        75
    positive       0.86      0.95      0.90       784

    accuracy                           0.82      1000
   macro avg       0.63      0.54      0.56      1000
weighted avg       0.80      0.82      0.80      1000



In [ ]:
# SVM (TF-IDF features) 
svm_clf = LinearSVC(max_iter=2000)
svm_clf.fit(X_train_tfidf, y_train)
svm_preds = svm_clf.predict(X_test_tfidf)

print('Classification Report — SVM / LinearSVC (TF-IDF):')
print(classification_report(y_test, svm_preds,
                             target_names=['negative', 'neutral', 'positive']))

Classification Report — SVM / LinearSVC (TF-IDF):
              precision    recall  f1-score   support

    negative       0.64      0.53      0.58       141
     neutral       0.22      0.05      0.09        75
    positive       0.86      0.95      0.90       784

    accuracy                           0.82      1000
   macro avg       0.58      0.51      0.52      1000
weighted avg       0.78      0.82      0.80      1000



## Step 5 — Model Comparison & Evaluation Summary

In [9]:
from sklearn.metrics import accuracy_score, f1_score

models = {
    'TextBlob':    tb_preds,
    'VADER':       vader_preds,
    'Naive Bayes': nb_preds,
    'SVM':         svm_preds,
}

summary = []
for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='weighted')
    summary.append([name, f'{acc:.4f}', f'{f1:.4f}'])

print(tabulate(summary,
               headers=['Model', 'Accuracy', 'Weighted F1'],
               tablefmt='grid'))

+-------------+------------+---------------+
| Model       |   Accuracy |   Weighted F1 |
+=============+============+===============+
| TextBlob    |      0.794 |        0.7596 |
+-------------+------------+---------------+
| VADER       |      0.788 |        0.74   |
+-------------+------------+---------------+
| Naive Bayes |      0.825 |        0.8043 |
+-------------+------------+---------------+
| SVM         |      0.825 |        0.7978 |
+-------------+------------+---------------+


## Step 6 — Export Extracted Data to CSV

In [10]:
# Save the preprocessed sample with sentiment labels
export_cols = ['Id', 'ProductId', 'UserId', 'ProfileName',
               'Score', 'Summary', 'Text', 'CleanText', 'Sentiment']
df[export_cols].to_csv('amazon_food_reviews_sentiment.csv', index=False)
print('Exported amazon_food_reviews_sentiment.csv')

Exported amazon_food_reviews_sentiment.csv
